# Massive Activation Perturbation Test: Token Identity vs. Positional Structure

**Hypothesis under test:** Sun et al. (COLM 2024) claim massive activations (MAs) are
*input-agnostic* — their values stay roughly constant regardless of input content. We test
a stronger, more specific version of this claim: is the MA at position 0 driven by the
**semantic identity** of the token occupying that position, or is it a **structural/positional**
effect that persists even when the position-0 token embedding is replaced by a random
direction of the same norm?

**Design**
1. Generate N random nonsensical sentences (real vocab tokens, no semantic structure), no BOS.
2. Run a baseline forward pass, confirm MAs replicate at position 0 / known channels
   (~788, 1384, 4062 for Llama-3.1-8B), consistent with prior notebook (`01f_*`).
3. Perturb the **position-0 input embedding** with a uniformly random direction on the
   hypersphere of the same norm (full direction randomization, exact norm preservation).
4. Re-run forward pass, compare MA magnitude/location pre- vs. post-perturbation.
5. Control: repeat the identical perturbation at a mid-sequence position instead of
   position 0, to confirm any effect is position-0-specific rather than generic.

**Interpretation:** if MA magnitude survives full direction randomization at fixed norm,
this is strong evidence MAs are structural/positional (input-agnostic in the strong sense),
supporting the confound-check framing (MAs as a control analysis, not a causal driver of
Jacobian spike behavior) rather than the reverse.

No BOS token is prepended anywhere in this notebook — position 0 is the first random
content token, consistent with the `01f_block_jacobian_svd_sweep.ipynb` setup.


## 1. Setup

In [ ]:
import os
import random
import json
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import pandas as pd

MODEL_PATH = "/home/samuel/research/llmattacks/llm-attacks/DIR/Llama-3.1-8B-Instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Device: {DEVICE}, dtype: {DTYPE}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=DTYPE,
    device_map=DEVICE,
)
model.eval()

VOCAB_SIZE = model.get_input_embeddings().weight.shape[0]
D_MODEL = model.get_input_embeddings().weight.shape[1]
N_LAYERS = model.config.num_hidden_layers

OUTPUT_DIR = "random_tokens_output"  # set to whatever path you want

print(f"Vocab size: {VOCAB_SIZE}, d_model: {D_MODEL}, n_layers: {N_LAYERS}")


## 2. Config

Known massive-activation channels and the reference position, carried over from
`01f_block_jacobian_svd_sweep.ipynb`. Adjust if the earlier notebook's findings change.

In [ ]:
N_SAMPLES = 10000          # number of random nonsensical sentences
SEQ_LEN = 2  # inclusive, tokens per sentence
PERTURB_POS = 0          # position 0 = first content token (no BOS)

KNOWN_MA_CHANNELS = [788, 1384, 4062]   # from prior notebook, replicated at position 0
MA_RATIO_THRESHOLD = 1000               # "genuine" massive activation threshold (Sun et al. convention)


## 3. Sample random nonsensical sentences

Token IDs are drawn uniformly from the vocabulary, excluding special/reserved tokens
(Llama-3.1's tokenizer reserves a large block of `<|reserved_special_token_*|>` and other
control tokens that would pollute both the "nonsensical sentence" semantics and the
activation statistics).

In [ ]:
def get_excluded_token_ids(tokenizer):
    excluded = set(tokenizer.all_special_ids)
    # Sweep the vocab once and flag anything that decodes to a reserved/control-looking
    # token, as a fallback in case all_special_ids doesn't capture every reserved slot.
    for tid in range(tokenizer.vocab_size, len(tokenizer.get_vocab())):
        excluded.add(tid)  # anything beyond base vocab_size is typically added/special
    return excluded

EXCLUDED_IDS = get_excluded_token_ids(tokenizer)
VALID_IDS = np.array([i for i in range(VOCAB_SIZE) if i not in EXCLUDED_IDS])
print(f"Excluded {len(EXCLUDED_IDS)} special/reserved ids; {len(VALID_IDS)} valid ids remain")


In [ ]:
def sample_random_sentence(rng):
    token_ids = rng.choice(VOCAB_SIZE, size=SEQ_LEN, replace=True).tolist()
    return token_ids

rng = np.random.default_rng(SEED)
samples = [sample_random_sentence(rng) for _ in range(N_SAMPLES)]

# Print decoded token lists for every sample as a sanity check.
# print("Decoded samples (sanity check):")
# for i, ids in enumerate(samples):
#     tokens = tokenizer.convert_ids_to_tokens(ids)
#     print(f"[{i:03d}] ids={ids}")
#     print(f"      tokens={tokens}")


## 5. Norm-preserving perturbation

For the token at the target position, replace its input embedding with a uniformly random
point on the hypersphere of the same norm:

$$e' = \|e\| \cdot \frac{n}{\|n\|}, \quad n \sim \mathcal{N}(0, I_d)$$

This is the full-randomization extreme case: in ~4096-d space, `e'` has expected cosine
similarity ≈ 0 with the original embedding `e`, while `||e'|| = ||e||` exactly.

In [ ]:
# Representative embedding norm: median over the full vocabulary's row norms.
# The specific token/vector attaining it is irrelevant — this is just a robust
# (outlier-resistant) scalar summary of "how big is a typical embedding vector".
embedding_matrix = model.get_input_embeddings().weight.detach().float()
embedding_row_norms = embedding_matrix[VALID_IDS].norm(dim=1)
MEDIAN_EMBEDDING_NORM = embedding_row_norms.median().item()

print(f"Embedding row norms — min={embedding_row_norms.min().item():.4f}  "
      f"median={MEDIAN_EMBEDDING_NORM:.4f}  max={embedding_row_norms.max().item():.4f}")


In [ ]:
def perturb_embedding_random_direction(embedding_row, generator, target_norm=None):
    """embedding_row: [d_model] tensor. Returns a new tensor with the given target_norm
    (default: the row's own norm) and a random direction.
    
    target_norm: scalar to scale the random unit vector to. Pass MEDIAN_EMBEDDING_NORM
    to scale to the vocabulary-wide median embedding norm instead of this token's own norm.
    """
    d = embedding_row.shape[0]
    noise = torch.randn(d, generator=generator, device=embedding_row.device, dtype=torch.float32)
    unit_noise = noise / noise.norm()

    if target_norm is None:
        target_norm = embedding_row.float().norm()

    new_row = unit_noise * target_norm
    return new_row.to(embedding_row.dtype)


def build_inputs_embeds_with_perturbation(input_ids, perturb_positions, torch_gen):
    """Return inputs_embeds [1, seq, d_model] with the tokens at perturb_positions replaced
    by random-direction, same-norm embeddings.

    perturb_positions: list/tuple of int positions to perturb (each gets an independent
    random direction, drawn from torch_gen).
    """
    embed_layer = model.get_input_embeddings()
    with torch.no_grad():
        base_embeds = embed_layer(input_ids).clone()  # [1, seq, d_model]
        for pos in perturb_positions:
            original_row = base_embeds[0, pos, :]
            perturbed_row = perturb_embedding_random_direction(original_row, torch_gen, target_norm=MEDIAN_EMBEDDING_NORM)
            base_embeds[0, pos, :] = perturbed_row
    return base_embeds


def build_inputs_embeds_no_perturbation(input_ids):
    """Return inputs_embeds [1, seq, d_model] with the token at perturb_position replaced
    by a random-direction, same-norm embedding."""
    embed_layer = model.get_input_embeddings()
    with torch.no_grad():
        base_embeds = embed_layer(input_ids).clone()  # [1, seq, d_model]
    return base_embeds


# Fresh, non-reproducible seed for the perturbation generator (independent of the
# fixed SEED above, which only controls sentence sampling) — each run picks a
# different random direction.
PERTURB_SEED = int.from_bytes(os.urandom(4), "big")
torch_gen = torch.Generator(device=DEVICE)
torch_gen.manual_seed(PERTURB_SEED)
print(f"torch_gen seeded with PERTURB_SEED={PERTURB_SEED}")

## 6. Perturbation experiment

In [ ]:
@torch.no_grad()
def run_forward(input_ids=None, inputs_embeds=None):
    """Run a forward pass from either input_ids or inputs_embeds, return hidden_states tuple."""
    if input_ids is not None:
        attention_mask = torch.ones_like(input_ids)
        out = model(input_ids=input_ids, attention_mask=attention_mask,
                     output_hidden_states=True, use_cache=False)
    else:
        attention_mask = torch.ones(inputs_embeds.shape[:2], dtype=torch.long, device=inputs_embeds.device)
        out = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask,
                     output_hidden_states=True, use_cache=False)
    # hidden_states: tuple of (n_layers + 1) tensors, each [batch, seq, d_model]
    return out.hidden_states

In [ ]:
perturb_positions = [0,1]
records = []

for i, ids in enumerate(samples):
    input_ids = torch.tensor([ids], device=DEVICE)
    inputs_embeds = build_inputs_embeds_with_perturbation(input_ids, perturb_positions, torch_gen)
    
    hidden_states = run_forward(inputs_embeds=inputs_embeds)

    seq_len = input_ids.shape[1]
    for layer, hs in enumerate(hidden_states):
        hs_0 = hs[0].float()                              # [seq_len, d_model]
        max_abs_per_pos = hs_0.abs().max(dim=-1).values    # [seq_len]
        argmax_per_pos = hs_0.abs().argmax(dim=-1)         # [seq_len]  <-- channel index of the max
        norm_per_pos = hs_0.norm(dim=-1)                   # [seq_len]
        for pos in range(seq_len):
            records.append({
                "sample": i,
                "layer": layer,
                "position": pos,
                "max_activation": max_abs_per_pos[pos].item(),
                "max_activation_channel": argmax_per_pos[pos].item(),
                "activation_norm": norm_per_pos[pos].item(),
            })


In [ ]:
# Saving the results and statistics
os.makedirs(OUTPUT_DIR, exist_ok=True)

per_position_df = pd.DataFrame(records)

layer_sum_df = (
    per_position_df.groupby(["sample", "layer"])["max_activation"]
    .sum()
    .reset_index()
    .rename(columns={"max_activation": "sum_max_activation"})
)

PER_POSITION_CSV = os.path.join(OUTPUT_DIR, f"ma_random_test_per_position_{N_SAMPLES}.csv")
LAYER_SUM_CSV = os.path.join(OUTPUT_DIR, f"ma_random_test_layer_sum_{N_SAMPLES}.csv")

per_position_df.to_csv(PER_POSITION_CSV, index=False)
layer_sum_df.to_csv(LAYER_SUM_CSV, index=False)

print(f"Saved per-position max activations -> {PER_POSITION_CSV} ({len(per_position_df)} rows)")
print(f"Saved per-layer sum of max activations -> {LAYER_SUM_CSV} ({len(layer_sum_df)} rows)")

## Additional Codes

In [ ]:
# per_position_df = pd.DataFrame(records)

# layer_sum_df = (
#     per_position_df.groupby(["sample", "layer"])["max_activation"]
#     .sum()
#     .reset_index()
#     .rename(columns={"max_activation": "sum_max_activation"})
# )

# PER_POSITION_CSV = "ma_random_test_per_position.csv"
# LAYER_SUM_CSV = "ma_random_test_layer_sum.csv"

# per_position_df.to_csv(PER_POSITION_CSV, index=False)
# layer_sum_df.to_csv(LAYER_SUM_CSV, index=False)

# print(f"Saved per-position max activations -> {PER_POSITION_CSV} ({len(per_position_df)} rows)")
# print(f"Saved per-layer sum of max activations -> {LAYER_SUM_CSV} ({len(layer_sum_df)} rows)")
